# Задание 4 — «Где живут наши Клиенты»

Адреса 605 клиентов автоцентра (Минск) наносим на карту в виде точек.

**Пайплайн:** геокодинг адресов через Nominatim (OpenStreetMap) → координаты → интерактивная карта folium (маркеры + тепловая карта плотности) → выводы.

Геокодинг вынесен в `geocode_addresses.py` + `retry_failures.py` (кэш в `geocoded.csv`), т.к. занимает ~10 мин. Ноутбук читает готовый кэш. Определено 524 из 584 уникальных адресов (90%).

Требуется: `pip install folium`

In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap

# все адреса клиентов (с повторами зданий)
df = pd.read_excel(r'..\task_4.xlsx')
# кэш координат по уникальным адресам (из geocode_addresses.py / retry_failures.py)
geo = pd.read_csv('geocoded.csv').dropna(subset=['lat', 'lon'])
geo['lat'] = geo['lat'].astype(float)
geo['lon'] = geo['lon'].astype(float)

# клиенты с координатами (дубли адресов сохраняются -> видно популярные дома/районы)
pts = df.merge(geo, on='Адрес', how='inner')
print(f'Адресов клиентов: {len(df)}')
print(f'Уникальных адресов: {df["Адрес"].nunique()}')
print(f'Геокодировано уникальных: {len(geo)} ({len(geo)/df["Адрес"].nunique()*100:.0f}%)')
print(f'Клиентов нанесено на карту: {len(pts)} ({len(pts)/len(df)*100:.0f}%)')

Адресов клиентов: 605
Уникальных адресов: 584
Геокодировано уникальных: 524 (90%)
Клиентов нанесено на карту: 543 (90%)


## Карта: точки клиентов (с кластеризацией)

In [2]:
MINSK = [53.9023, 27.5619]

m = folium.Map(location=MINSK, zoom_start=11, tiles='OpenStreetMap')
cluster = MarkerCluster().add_to(m)
for _, r in pts.iterrows():
    folium.CircleMarker(
        [r['lat'], r['lon']], radius=4, color='#d62728',
        fill=True, fill_color='#d62728', fill_opacity=0.7,
        popup=str(r['Адрес'])
    ).add_to(cluster)

m.save('minsk_clients_map.html')
print('Сохранено: minsk_clients_map.html')
m

Сохранено: minsk_clients_map.html


## Тепловая карта плотности клиентов

In [3]:
h = folium.Map(location=MINSK, zoom_start=11, tiles='CartoDB positron')
HeatMap(pts[['lat', 'lon']].values.tolist(), radius=13, blur=18).add_to(h)
h.save('minsk_clients_heatmap.html')
print('Сохранено: minsk_clients_heatmap.html')
h

Сохранено: minsk_clients_heatmap.html


## Распределение относительно центра города

In [4]:
import numpy as np

lat0, lon0 = MINSK
dlat = (pts['lat'] - lat0) * 111.0
dlon = (pts['lon'] - lon0) * 111.0 * np.cos(np.radians(lat0))
pts = pts.assign(dist_km=np.hypot(dlat, dlon))

print('Расстояние клиентов от центра, км:')
print(pts['dist_km'].describe().round(2))
for r in (3, 5, 8):
    print(f'  в пределах {r} км от центра: {(pts["dist_km"] <= r).mean()*100:.0f}%')
print(f'  восточнее центра (lon > {lon0}): {(pts["lon"] > lon0).mean()*100:.0f}%')

print('\nЧаще всего повторяющиеся адреса (дома с макс. числом клиентов):')
print(df['Адрес'].value_counts().head(8))

Расстояние клиентов от центра, км:
count    543.00
mean       6.76
std        2.43
min        0.52
25%        5.22
50%        7.48
75%        8.65
max       15.80
Name: dist_km, dtype: float64
  в пределах 3 км от центра: 10%
  в пределах 5 км от центра: 24%
  в пределах 8 км от центра: 61%
  восточнее центра (lon > 27.5619): 17%

Чаще всего повторяющиеся адреса (дома с макс. числом клиентов):
Адрес
пр-т. газ. "Правда", 22,     3
пр-т. Дзержинского, д.119    2
пр-т. Дзержинского, д.131    2
Папанина, д.15               2
Слободская, 45               2
Космонавтов, 37,             2
пр-т. Дзержинского, 9,       2
Ландера, 10,                 2
Name: count, dtype: int64


## Выводы

**Данные карты:** нанесено 543 из 605 клиентов (90%), геокодинг Nominatim/OSM.

**1. Клиенты живут не в центре, а в спальных районах.** Медианное расстояние от центра ≈ 7.5 км. В пределах 3 км от центра — лишь 10% клиентов, 5 км — 24%, 8 км — 61%. Ядро клиентской базы — жители периферийного жилого пояса.

**2. Выраженная концентрация на юго-западе/западе Минска.** Только 17% клиентов живут восточнее центра. Самые частые адреса — проспект Дзержинского (дома 9, 119, 131), пр-т газ. «Правда», Ландера, Слободская, Космонавтов — это юго-западные районы (Малиновка, Юго-Запад, Курасовщина).

**3. Гипотеза о локации автоцентра.** Такая кластеризация типична, когда клиентская база формируется по географической близости к точке продаж. Вероятно, автоцентр расположен на юго-западе/западе города — покупатели преимущественно из окрестных микрорайонов.

### Практические выводы для бизнеса
- **Локальный маркетинг** (наружная реклама, листовки, геотаргетинг) эффективнее всего в юго-западных районах — там основная масса клиентов.
- **Восток и северо-восток** города — зона недопредставленности: либо там сильны конкуренты, либо не доходит реклама. Это потенциал для роста (тестовые кампании).
- Для клиентов из отдалённых районов (медиана 7.5 км) важны сервисные удобства: подменный автомобиль, выездной сервис, трейд-ин — снижают барьер расстояния.

### Ограничения
- ~10% адресов не геокодированы (нестандартная запись, отсутствие дома в OSM) — карта слегка недооценивает плотность.
- Nominatim может ставить точку в центроид улицы при неточном номере дома — на уровне районов выводы устойчивы, точечная привязка отдельных адресов приблизительна.